# 📊 Pipeline: Data Cleaning → SQLite → Visualisasi
### Data Survei Persepsi Mahasiswa & Review Canva (Google Play)

**Alur Kerja:**
1. Upload & baca data mentah
2. Pembersihan data (duplikat, NaN, format teks via Regex)
3. Simpan ke basis data relasional SQLite3
4. Query SQL (SELECT, WHERE)
5. Visualisasi data bersih

---
## ⚙️ CELL 1 — Install Library yang Diperlukan

In [ ]:
!pip install google-play-scraper pandas openpyxl matplotlib seaborn wordcloud --quiet
print("✅ Semua library berhasil diinstall")

---
## 📁 CELL 2 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import re
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
from google_play_scraper import reviews, Sort
from google.colab import files

# Style plot global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print("✅ Semua library berhasil diimport")

---
# ═══════════════════════════════════════════
# BAGIAN A — DATA SURVEI PERSEPSI MAHASISWA
# ═══════════════════════════════════════════

## 📂 CELL 3 — Upload & Baca Data Survei (CSV)

In [ ]:
print("📤 Silakan upload file CSV survei (Form Responses)...")
uploaded = files.upload()

# Ambil nama file yang diupload
survey_filename = list(uploaded.keys())[0]
print(f"\n✅ File diterima: {survey_filename}")

# Baca CSV
df_survey_raw = pd.read_csv(survey_filename)
print(f"\n📊 Shape data mentah: {df_survey_raw.shape}")
print("\n🔍 5 baris pertama:")
display(df_survey_raw.head())

## 🧹 CELL 4 — Pembersihan Data Survei
### (Hapus Duplikat, Tangani NaN, Rapikan Format Teks via Regex)

In [ ]:
df_survey = df_survey_raw.copy()

print("=" * 55)
print("🔎 LAPORAN DATA MENTAH")
print("=" * 55)
print(f"Total baris    : {len(df_survey)}")
print(f"Total kolom    : {df_survey.shape[1]}")
print(f"Data duplikat  : {df_survey.duplicated().sum()}")
print(f"Total nilai NaN: {df_survey.isnull().sum().sum()}")
print()
print("Nilai NaN per kolom:")
print(df_survey.isnull().sum())

# ── 1. Hapus baris duplikat penuh ──────────────────────────
sebelum = len(df_survey)
df_survey.drop_duplicates(inplace=True)
print(f"\n✅ [1] Duplikat dihapus: {sebelum - len(df_survey)} baris")

# ── 2. Tangani NaN ─────────────────────────────────────────
# Kolom teks kategorik: isi dengan modus
kolom_kategorik = ['Jenis Kelamin', 'Program Studi',
                   'Apakah Anda pernah menggunakan Canva?',
                   'Frekuensi Penggunaan Canva']

for col in kolom_kategorik:
    if col in df_survey.columns and df_survey[col].isnull().sum() > 0:
        modus = df_survey[col].mode()[0]
        df_survey[col].fillna(modus, inplace=True)
        print(f"   → NaN di '{col}' diisi dengan modus: '{modus}'")

# Kolom Likert (skala 1-5 / teks persetujuan): isi dengan modus
kolom_likert = [c for c in df_survey.columns
                if c not in ['Timestamp', 'Email Address', 'Nama',
                             'Usia', 'Jenis Kelamin', 'Program Studi',
                             'Apakah Anda pernah menggunakan Canva?',
                             'Frekuensi Penggunaan Canva']]
for col in kolom_likert:
    if col in df_survey.columns and df_survey[col].isnull().sum() > 0:
        modus = df_survey[col].mode()[0]
        df_survey[col].fillna(modus, inplace=True)

# Kolom Usia: isi dengan median
if df_survey['Usia'].isnull().sum() > 0:
    median_usia = df_survey['Usia'].median()
    df_survey['Usia'].fillna(median_usia, inplace=True)
    print(f"   → NaN di 'Usia' diisi dengan median: {median_usia}")

print(f"\n✅ [2] Penanganan NaN selesai. Sisa NaN: {df_survey.isnull().sum().sum()}")

# ── 3. Bersihkan format teks via Regex ─────────────────────
def bersihkan_teks(teks):
    """Rapikan whitespace berlebih dan karakter tak terlihat."""
    if isinstance(teks, str):
        teks = re.sub(r'[\t\r\n]+', ' ', teks)   # tab/newline → spasi
        teks = re.sub(r' {2,}', ' ', teks)        # spasi ganda → 1 spasi
        teks = re.sub(r'^\s+|\s+$', '', teks)     # strip leading/trailing
        teks = re.sub(r'[\x00-\x1F\x7F]', '', teks)  # hapus kontrol chars
    return teks

kolom_teks_survei = ['Nama', 'Program Studi', 'Jenis Kelamin',
                     'Apakah Anda pernah menggunakan Canva?',
                     'Frekuensi Penggunaan Canva']

for col in kolom_teks_survei:
    if col in df_survey.columns:
        df_survey[col] = df_survey[col].apply(bersihkan_teks)

# Normalisasi Jenis Kelamin (title case)
if 'Jenis Kelamin' in df_survey.columns:
    df_survey['Jenis Kelamin'] = df_survey['Jenis Kelamin'].str.strip().str.title()

# Normalisasi Program Studi (title case)
if 'Program Studi' in df_survey.columns:
    df_survey['Program Studi'] = df_survey['Program Studi'].str.strip().str.title()

# Konversi Timestamp ke datetime
df_survey['Timestamp'] = pd.to_datetime(df_survey['Timestamp'],
                                         format='%d/%m/%Y %H:%M:%S',
                                         errors='coerce')

# Konversi Usia ke int (jika memungkinkan)
df_survey['Usia'] = pd.to_numeric(df_survey['Usia'], errors='coerce').fillna(0).astype(int)

print("✅ [3] Pembersihan format teks (Regex) selesai")

# ── Rename kolom agar singkat dan cocok untuk SQLite ───────
rename_map = {
    'Timestamp': 'timestamp',
    'Email Address': 'email',
    'Nama': 'nama',
    'Usia': 'usia',
    'Jenis Kelamin': 'jenis_kelamin',
    'Program Studi': 'program_studi',
    'Apakah Anda pernah menggunakan Canva?': 'pernah_pakai_canva',
    'Frekuensi Penggunaan Canva': 'frekuensi_penggunaan',
    'Penggunaan Canva dalam project tugas kuliah menjadi lebih praktis dan cepat': 'q_praktis_cepat',
    'Canva menawarkan fitur premium gratis bagi dosen dan mahasiswa ': 'q_fitur_premium_gratis',
    'Canva membantu meningkatkan kualitas visual tugas atau materi yang saya buat': 'q_kualitas_visual',
    'Canva mempermudah saya dalam menyelesaikan berbagai jenis tugas (presentasi, poster, infografis, dll.)': 'q_mempermudah_tugas',
    'Penggunaan Canva membuat hasil pekerjaan saya terlihat lebih profesional': 'q_terlihat_profesional',
    'Membuat pembelajaran lebih interaktif dan kreatif': 'q_interaktif_kreatif',
    'Fitur-fitur Canva mudah digunakan untuk membuat materi pembelajaran atau tugas': 'q_fitur_mudah',
    'Navigasi dalam Canva sederhana sehingga mempermudah proses pembuatan konten pembelajaran': 'q_navigasi_sederhana',
    'Saya dapat dengan mudah menemukan fitur yang saya butuhkan di Canva tanpa kesulitan': 'q_mudah_temukan_fitur',
    'Saya tidak memerlukan waktu lama untuk mempelajari cara menggunakan Canva secara efektif': 'q_cepat_belajar',
    'Canva memberikan rasa aman dalam mengelola data tugas dan proyek pembelajaran': 'q_rasa_aman',
    'Saya percaya Canva merupakan platform yang dapat diandalkan untuk keperluan akademik.': 'q_platform_andal',
    'Saya percaya Canva menjaga privasi data pengguna dengan baik.': 'q_privasi_data',
    'Canva jarang mengalami masalah teknis yang mengganggu pekerjaan saya.': 'q_jarang_error',
    'Reputasi Canva yang sudah sangat populer di berbagai kalangan membuat saya merasa aman dalam menggunakannya.': 'q_reputasi_populer',
    'Saya akan menggunakan aplikasi Canva secara berulang untuk kebutuhan pembelajaran': 'q_pakai_berulang',
    'Saya akan merekomendasikan aplikasi canva kepada teman belajar saya': 'q_rekomendasikan',
    'Saya akan memilih Canva dibandingkan dengan aplikasi untuk membuat materi pembelajaran.': 'q_pilih_canva',
    'Saya tertarik mengeksplorasi lebih banyak fitur Canva untuk mendukung pembelajaran.': 'q_eksplorasi_fitur',
    'Saya berniat menjadikan Canva sebagai salah satu aplikasi utama dalam mendukung kegiatan pembelajaran saya': 'q_jadikan_utama'
}

df_survey.rename(columns={k: v for k, v in rename_map.items() if k in df_survey.columns},
                 inplace=True)

# ── Konversi Likert teks → angka ───────────────────────────
likert_map = {
    'Sangat Tidak Setuju': 1, 'Tidak Setuju': 2,
    'Netral': 3, 'Setuju': 4, 'Sangat Setuju': 5
}

kolom_q = [c for c in df_survey.columns if c.startswith('q_')]
for col in kolom_q:
    df_survey[col] = df_survey[col].map(likert_map)

# Tambah kolom ID unik
df_survey.insert(0, 'id_responden', range(1, len(df_survey) + 1))

print(f"\n=" * 55)
print("✅ DATA SURVEI BERSIH")
print(f"=" * 55)
print(f"Total baris  : {len(df_survey)}")
print(f"Total kolom  : {df_survey.shape[1]}")
print(f"Sisa NaN     : {df_survey.isnull().sum().sum()}")
print()
display(df_survey.head(3))

---
# ═══════════════════════════════════════════
# BAGIAN B — DATA REVIEW ULASAN CANVA (SCRAPING)
# ═══════════════════════════════════════════

## 🌐 CELL 5 — Scraping Review Canva dari Google Play (Maks 200 Data)

In [ ]:
print("🔄 Mengambil review Canva dari Google Play Store (maks 200 data)...")

all_reviews = []
token = None
target = 200

while len(all_reviews) < target:
    batch, token = reviews(
        'com.canva.editor',
        lang='id',
        country='id',
        sort=Sort.NEWEST,
        count=100,
        continuation_token=token
    )
    all_reviews.extend(batch)
    print(f"  Terkumpul: {len(all_reviews)} review")
    if token is None or len(batch) == 0:
        break

# Ambil tepat 200
all_reviews = all_reviews[:target]

df_review_raw = pd.DataFrame(all_reviews)
df_review_raw = df_review_raw[['userName', 'score', 'content',
                                'thumbsUpCount', 'reviewCreatedVersion', 'at']]
df_review_raw.columns = ['Username', 'Rating', 'Review',
                          'Likes', 'Versi_Aplikasi', 'Tanggal']

print(f"\n✅ Scraping selesai. Total: {len(df_review_raw)} review")
display(df_review_raw.head())

## 🧹 CELL 6 — Pembersihan Data Review
### (Hapus Duplikat, Tangani NaN, Rapikan Teks via Regex)

In [ ]:
df_review = df_review_raw.copy()

print("=" * 55)
print("🔎 LAPORAN DATA REVIEW MENTAH")
print("=" * 55)
print(f"Total baris    : {len(df_review)}")
print(f"Data duplikat  : {df_review.duplicated().sum()}")
print(f"Total NaN      : {df_review.isnull().sum().sum()}")
print()
print(df_review.isnull().sum())

# ── 1. Hapus duplikat ──────────────────────────────────────
sebelum = len(df_review)
df_review.drop_duplicates(inplace=True)
print(f"\n✅ [1] Duplikat dihapus: {sebelum - len(df_review)} baris")

# ── 2. Tangani NaN ─────────────────────────────────────────
# Review kosong → hapus baris (review adalah inti data)
sebelum = len(df_review)
df_review.dropna(subset=['Review'], inplace=True)
print(f"✅ [2a] Baris dengan Review kosong dihapus: {sebelum - len(df_review)}")

# Username kosong → isi 'Anonim'
df_review['Username'].fillna('Anonim', inplace=True)

# Likes kosong → isi 0
df_review['Likes'].fillna(0, inplace=True)

# Versi Aplikasi kosong → isi 'Tidak Diketahui'
df_review['Versi_Aplikasi'].fillna('Tidak Diketahui', inplace=True)

# Rating kosong → isi median
if df_review['Rating'].isnull().sum() > 0:
    median_rating = df_review['Rating'].median()
    df_review['Rating'].fillna(median_rating, inplace=True)

print(f"✅ [2b] NaN lainnya ditangani. Sisa NaN: {df_review.isnull().sum().sum()}")

# ── 3. Pembersihan teks via Regex ──────────────────────────
def bersihkan_review(teks):
    """Pipeline pembersihan teks ulasan."""
    if not isinstance(teks, str):
        return ''
    # Hapus URL
    teks = re.sub(r'https?://\S+|www\.\S+', '', teks)
    # Hapus karakter kontrol & emoji (non-ASCII opsional: komentari jika mau simpan emoji)
    teks = re.sub(r'[\x00-\x1F\x7F]', '', teks)
    # Hapus karakter non-alfanumerik kecuali tanda baca umum
    teks = re.sub(r'[^\w\s.,!?;:\'"-]', ' ', teks, flags=re.UNICODE)
    # Ganti newline & tab dengan spasi
    teks = re.sub(r'[\n\r\t]+', ' ', teks)
    # Hapus spasi berlebih
    teks = re.sub(r' {2,}', ' ', teks).strip()
    # Hapus angka berulang panjang (noise)
    teks = re.sub(r'\b\d{8,}\b', '', teks)
    return teks

df_review['Review_Bersih'] = df_review['Review'].apply(bersihkan_review)

# Bersihkan Username
def bersihkan_username(nama):
    if not isinstance(nama, str):
        return 'Anonim'
    nama = re.sub(r'[\x00-\x1F\x7F]', '', nama)
    nama = re.sub(r' {2,}', ' ', nama).strip()
    return nama if nama else 'Anonim'

df_review['Username'] = df_review['Username'].apply(bersihkan_username)

# Bersihkan Versi Aplikasi
df_review['Versi_Aplikasi'] = df_review['Versi_Aplikasi'].apply(
    lambda x: re.sub(r'[^\d.]', '', str(x)) if isinstance(x, str) else 'Tidak Diketahui'
)
df_review['Versi_Aplikasi'] = df_review['Versi_Aplikasi'].replace('', 'Tidak Diketahui')

# Konversi tipe data
df_review['Rating'] = df_review['Rating'].astype(int)
df_review['Likes'] = df_review['Likes'].astype(int)
df_review['Tanggal'] = pd.to_datetime(df_review['Tanggal'], errors='coerce')
df_review['Tanggal_Str'] = df_review['Tanggal'].dt.strftime('%Y-%m-%d')

# Kolom sentimen sederhana berdasarkan rating
def sentimen(rating):
    if rating >= 4:
        return 'Positif'
    elif rating == 3:
        return 'Netral'
    else:
        return 'Negatif'

df_review['Sentimen'] = df_review['Rating'].apply(sentimen)

# Tambah ID unik
df_review.insert(0, 'id_review', range(1, len(df_review) + 1))

# Hapus review kosong setelah cleaning
df_review = df_review[df_review['Review_Bersih'].str.len() > 5]

# Pastikan maks 200
df_review = df_review.head(200).reset_index(drop=True)

print(f"\n=" * 55)
print("✅ DATA REVIEW BERSIH")
print(f"=" * 55)
print(f"Total baris  : {len(df_review)}")
print(f"Sisa NaN     : {df_review.isnull().sum().sum()}")
print()
display(df_review[['id_review','Username','Rating','Review_Bersih','Sentimen','Tanggal_Str']].head(5))

---
# ═══════════════════════════════════════════
# BAGIAN C — BASIS DATA SQLITE3
# ═══════════════════════════════════════════

## 🗄️ CELL 7 — Buat & Isi Database SQLite3

In [ ]:
DB_PATH = 'canva_data.db'

# Hapus DB lama jika ada
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# ── Buat tabel survei ──────────────────────────────────────
cursor.execute("""
CREATE TABLE IF NOT EXISTS survei_responden (
    id_responden        INTEGER PRIMARY KEY,
    timestamp           TEXT,
    email               TEXT,
    nama                TEXT,
    usia                INTEGER,
    jenis_kelamin       TEXT,
    program_studi       TEXT,
    pernah_pakai_canva  TEXT,
    frekuensi_penggunaan TEXT,
    q_praktis_cepat          INTEGER,
    q_fitur_premium_gratis   INTEGER,
    q_kualitas_visual        INTEGER,
    q_mempermudah_tugas      INTEGER,
    q_terlihat_profesional   INTEGER,
    q_interaktif_kreatif     INTEGER,
    q_fitur_mudah            INTEGER,
    q_navigasi_sederhana     INTEGER,
    q_mudah_temukan_fitur    INTEGER,
    q_cepat_belajar          INTEGER,
    q_rasa_aman              INTEGER,
    q_platform_andal         INTEGER,
    q_privasi_data           INTEGER,
    q_jarang_error           INTEGER,
    q_reputasi_populer       INTEGER,
    q_pakai_berulang         INTEGER,
    q_rekomendasikan         INTEGER,
    q_pilih_canva            INTEGER,
    q_eksplorasi_fitur       INTEGER,
    q_jadikan_utama          INTEGER
)
""")

# ── Buat tabel review ──────────────────────────────────────
cursor.execute("""
CREATE TABLE IF NOT EXISTS review_ulasan (
    id_review       INTEGER PRIMARY KEY,
    username        TEXT,
    rating          INTEGER,
    review_asli     TEXT,
    review_bersih   TEXT,
    likes           INTEGER,
    versi_aplikasi  TEXT,
    tanggal         TEXT,
    sentimen        TEXT
)
""")

conn.commit()
print("✅ Tabel 'survei_responden' dan 'review_ulasan' berhasil dibuat")

# ── Insert data survei ─────────────────────────────────────
kolom_survei = [
    'id_responden', 'timestamp', 'email', 'nama', 'usia',
    'jenis_kelamin', 'program_studi', 'pernah_pakai_canva', 'frekuensi_penggunaan',
    'q_praktis_cepat', 'q_fitur_premium_gratis', 'q_kualitas_visual',
    'q_mempermudah_tugas', 'q_terlihat_profesional', 'q_interaktif_kreatif',
    'q_fitur_mudah', 'q_navigasi_sederhana', 'q_mudah_temukan_fitur',
    'q_cepat_belajar', 'q_rasa_aman', 'q_platform_andal',
    'q_privasi_data', 'q_jarang_error', 'q_reputasi_populer',
    'q_pakai_berulang', 'q_rekomendasikan', 'q_pilih_canva',
    'q_eksplorasi_fitur', 'q_jadikan_utama'
]

# Filter hanya kolom yang ada
kolom_survei_ada = [c for c in kolom_survei if c in df_survey.columns]
df_survey_insert = df_survey[kolom_survei_ada].copy()
df_survey_insert['timestamp'] = df_survey_insert['timestamp'].astype(str)
df_survey_insert.to_sql('survei_responden', conn, if_exists='replace', index=False)
print(f"✅ {len(df_survey_insert)} baris survei berhasil dimasukkan ke DB")

# ── Insert data review ─────────────────────────────────────
df_review_insert = df_review[[
    'id_review', 'Username', 'Rating', 'Review', 'Review_Bersih',
    'Likes', 'Versi_Aplikasi', 'Tanggal_Str', 'Sentimen'
]].copy()
df_review_insert.columns = [
    'id_review', 'username', 'rating', 'review_asli', 'review_bersih',
    'likes', 'versi_aplikasi', 'tanggal', 'sentimen'
]
df_review_insert.to_sql('review_ulasan', conn, if_exists='replace', index=False)
print(f"✅ {len(df_review_insert)} baris review berhasil dimasukkan ke DB")

print(f"\n🗄️  Database tersimpan di: {DB_PATH}")

## 🔍 CELL 8 — Query SQL: SELECT & WHERE

In [ ]:
print("=" * 60)
print("📋 QUERY SQL — DATA SURVEI")
print("=" * 60)

# Query 1: Semua responden
q1 = "SELECT id_responden, nama, usia, jenis_kelamin, program_studi, frekuensi_penggunaan FROM survei_responden"
df_q1 = pd.read_sql_query(q1, conn)
print(f"\n🔹 Query 1 — Semua Responden ({len(df_q1)} baris):")
print(q1)
display(df_q1.head(5))

# Query 2: Responden yang setuju Canva praktis (skor >= 4)
q2 = "SELECT nama, program_studi, q_praktis_cepat, q_kualitas_visual FROM survei_responden WHERE q_praktis_cepat >= 4"
df_q2 = pd.read_sql_query(q2, conn)
print(f"\n🔹 Query 2 — Responden: Canva Praktis & Cepat (skor ≥ 4) → {len(df_q2)} baris:")
print(q2)
display(df_q2.head(5))

# Query 3: Rata-rata skor per aspek
q3 = """
SELECT
    ROUND(AVG(q_praktis_cepat), 2)        AS avg_praktis_cepat,
    ROUND(AVG(q_kualitas_visual), 2)      AS avg_kualitas_visual,
    ROUND(AVG(q_fitur_mudah), 2)          AS avg_fitur_mudah,
    ROUND(AVG(q_platform_andal), 2)       AS avg_platform_andal,
    ROUND(AVG(q_rekomendasikan), 2)       AS avg_rekomendasikan
FROM survei_responden
"""
df_q3 = pd.read_sql_query(q3, conn)
print(f"\n🔹 Query 3 — Rata-rata Skor per Aspek Utama:")
print(q3.strip())
display(df_q3)

# Query 4: Distribusi frekuensi penggunaan
q4 = """
SELECT frekuensi_penggunaan, COUNT(*) AS jumlah
FROM survei_responden
GROUP BY frekuensi_penggunaan
ORDER BY jumlah DESC
"""
df_q4 = pd.read_sql_query(q4, conn)
print(f"\n🔹 Query 4 — Distribusi Frekuensi Penggunaan:")
print(q4.strip())
display(df_q4)

print("\n" + "=" * 60)
print("📋 QUERY SQL — DATA REVIEW ULASAN")
print("=" * 60)

# Query 5: Semua review bersih
q5 = "SELECT id_review, username, rating, sentimen, review_bersih FROM review_ulasan LIMIT 5"
df_q5 = pd.read_sql_query(q5, conn)
print(f"\n🔹 Query 5 — Sample Data Review Bersih:")
print(q5)
display(df_q5)

# Query 6: Review positif (rating >= 4)
q6 = "SELECT username, rating, review_bersih, likes FROM review_ulasan WHERE sentimen = 'Positif' ORDER BY likes DESC LIMIT 5"
df_q6 = pd.read_sql_query(q6, conn)
print(f"\n🔹 Query 6 — Review Positif (Rating ≥ 4), Diurutkan Likes Terbanyak:")
print(q6)
display(df_q6)

# Query 7: Distribusi sentimen
q7 = """
SELECT sentimen, COUNT(*) AS jumlah,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM review_ulasan), 1) AS persen
FROM review_ulasan
GROUP BY sentimen
ORDER BY jumlah DESC
"""
df_q7 = pd.read_sql_query(q7, conn)
print(f"\n🔹 Query 7 — Distribusi Sentimen:")
print(q7.strip())
display(df_q7)

# Query 8: Review negatif atau netral
q8 = """
SELECT username, rating, sentimen, review_bersih
FROM review_ulasan
WHERE sentimen IN ('Negatif', 'Netral')
ORDER BY rating ASC
LIMIT 5
"""
df_q8 = pd.read_sql_query(q8, conn)
print(f"\n🔹 Query 8 — Review Negatif & Netral:")
print(q8.strip())
display(df_q8)

---
# ═══════════════════════════════════════════
# BAGIAN D — VISUALISASI
# ═══════════════════════════════════════════

## 📊 CELL 9 — Visualisasi Data Survei

In [ ]:
# Ambil data bersih dari SQLite untuk visualisasi
df_viz_survei = pd.read_sql_query("SELECT * FROM survei_responden", conn)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('📊 Visualisasi Data Survei Persepsi Mahasiswa terhadap Canva',
             fontsize=15, fontweight='bold', y=1.01)

PALETTE = ['#4C9BE8', '#E8834C', '#4CE882', '#E84C6B', '#9B4CE8', '#E8C84C']

# ── Plot 1: Distribusi Jenis Kelamin (Pie) ─────────────────
ax1 = axes[0, 0]
jk_counts = df_viz_survei['jenis_kelamin'].value_counts()
ax1.pie(jk_counts, labels=jk_counts.index, autopct='%1.1f%%',
        colors=PALETTE[:len(jk_counts)], startangle=140,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax1.set_title('Distribusi Jenis Kelamin', fontweight='bold')

# ── Plot 2: Distribusi Program Studi (Bar Horizontal) ──────
ax2 = axes[0, 1]
prodi = df_viz_survei['program_studi'].value_counts().head(8)
bars = ax2.barh(prodi.index[::-1], prodi.values[::-1], color=PALETTE[0])
ax2.bar_label(bars, padding=3, fontsize=9)
ax2.set_xlabel('Jumlah')
ax2.set_title('Top Program Studi Responden', fontweight='bold')
ax2.set_xlim(0, prodi.max() + 5)

# ── Plot 3: Frekuensi Penggunaan Canva (Bar) ───────────────
ax3 = axes[0, 2]
freq = df_viz_survei['frekuensi_penggunaan'].value_counts()
bars3 = ax3.bar(freq.index, freq.values, color=PALETTE[1], edgecolor='white')
ax3.bar_label(bars3, padding=3, fontsize=9)
ax3.set_xlabel('Frekuensi')
ax3.set_ylabel('Jumlah Responden')
ax3.set_title('Frekuensi Penggunaan Canva', fontweight='bold')
ax3.tick_params(axis='x', rotation=20)

# ── Plot 4: Rata-rata Skor per Dimensi (Radar/Bar) ─────────
ax4 = axes[1, 0]
dimensi = {
    'Manfaat\nPembelajaran': ['q_praktis_cepat','q_kualitas_visual',
                              'q_mempermudah_tugas','q_terlihat_profesional',
                              'q_interaktif_kreatif'],
    'Kemudahan\nPenggunaan': ['q_fitur_mudah','q_navigasi_sederhana',
                              'q_mudah_temukan_fitur','q_cepat_belajar'],
    'Kepercayaan\n& Keamanan': ['q_rasa_aman','q_platform_andal',
                                'q_privasi_data','q_jarang_error',
                                'q_reputasi_populer'],
    'Minat\nBerlanjut':       ['q_pakai_berulang','q_rekomendasikan',
                               'q_pilih_canva','q_eksplorasi_fitur',
                               'q_jadikan_utama']
}
dim_scores = {}
for dim, cols in dimensi.items():
    valid_cols = [c for c in cols if c in df_viz_survei.columns]
    dim_scores[dim] = df_viz_survei[valid_cols].mean().mean()

labels_dim = list(dim_scores.keys())
values_dim = list(dim_scores.values())
bars4 = ax4.bar(labels_dim, values_dim, color=PALETTE[2:6], edgecolor='white')
ax4.bar_label(bars4, fmt='%.2f', padding=3, fontsize=10)
ax4.set_ylim(1, 5.5)
ax4.axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Netral (3.0)')
ax4.set_ylabel('Rata-rata Skor Likert (1–5)')
ax4.set_title('Rata-rata Skor per Dimensi', fontweight='bold')
ax4.legend(fontsize=8)

# ── Plot 5: Distribusi Usia Responden (Histogram) ──────────
ax5 = axes[1, 1]
ax5.hist(df_viz_survei['usia'].dropna(), bins=8, color=PALETTE[4],
         edgecolor='white', linewidth=1.2)
ax5.set_xlabel('Usia')
ax5.set_ylabel('Jumlah Responden')
ax5.set_title('Distribusi Usia Responden', fontweight='bold')
ax5.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# ── Plot 6: Perbandingan Skor Tiap Pertanyaan (Heatmap) ────
ax6 = axes[1, 2]
kolom_q_viz = [c for c in df_viz_survei.columns if c.startswith('q_')]
avg_q = df_viz_survei[kolom_q_viz].mean().reset_index()
avg_q.columns = ['Pertanyaan', 'Rata-rata']
avg_q['Pertanyaan'] = avg_q['Pertanyaan'].str.replace('q_', '').str.replace('_', ' ').str.title()

# Horizontal bar untuk skor tiap pertanyaan
colors_bar = ['#E84C4C' if v < 3.5 else '#4C9BE8' if v < 4 else '#4CE882'
               for v in avg_q['Rata-rata']]
bars6 = ax6.barh(avg_q['Pertanyaan'], avg_q['Rata-rata'], color=colors_bar)
ax6.axvline(x=4, color='gray', linestyle='--', alpha=0.5, label='Skor 4.0')
ax6.set_xlim(1, 5.5)
ax6.set_xlabel('Rata-rata Skor')
ax6.set_title('Rata-rata Skor Setiap Pertanyaan', fontweight='bold')
ax6.legend(fontsize=8)
ax6.tick_params(axis='y', labelsize=7.5)

plt.tight_layout()
plt.savefig('visualisasi_survei.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Visualisasi survei tersimpan: visualisasi_survei.png")

## 📊 CELL 10 — Visualisasi Data Review Ulasan Canva (Maks 200 Data)

In [ ]:
# Ambil maks 200 data review dari SQLite
df_viz_review = pd.read_sql_query(
    "SELECT * FROM review_ulasan ORDER BY id_review LIMIT 200", conn
)

fig2, axes2 = plt.subplots(2, 3, figsize=(18, 11))
fig2.suptitle('🌟 Visualisasi Data Review Ulasan Canva — Google Play (Maks 200 Data)',
              fontsize=15, fontweight='bold', y=1.01)

PALETTE2 = {'Positif': '#4CE882', 'Netral': '#E8C84C', 'Negatif': '#E84C4C'}

# ── Plot 1: Distribusi Sentimen (Donut Chart) ───────────────
ax = axes2[0, 0]
sent_counts = df_viz_review['sentimen'].value_counts()
colors_sent = [PALETTE2.get(s, '#AAAAAA') for s in sent_counts.index]
wedges, texts, autotexts = ax.pie(
    sent_counts, labels=sent_counts.index, autopct='%1.1f%%',
    colors=colors_sent, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2, 'width': 0.6}
)
ax.set_title('Distribusi Sentimen Review', fontweight='bold')
# Tambah teks jumlah di tengah donut
ax.text(0, 0, f'{len(df_viz_review)}\nReview', ha='center', va='center',
        fontsize=11, fontweight='bold')

# ── Plot 2: Distribusi Rating (Bar) ────────────────────────
ax2b = axes2[0, 1]
rating_counts = df_viz_review['rating'].value_counts().sort_index()
rating_colors = ['#E84C4C','#E87C4C','#E8C84C','#84E84C','#4CE882']
bars2b = ax2b.bar(rating_counts.index.astype(str), rating_counts.values,
                  color=rating_colors[:len(rating_counts)], edgecolor='white')
ax2b.bar_label(bars2b, padding=3, fontsize=10)
ax2b.set_xlabel('Rating Bintang')
ax2b.set_ylabel('Jumlah Review')
ax2b.set_title('Distribusi Rating (1–5 Bintang)', fontweight='bold')

# ── Plot 3: Trend Review per Bulan (Line) ──────────────────
ax3b = axes2[0, 2]
df_viz_review['tanggal_dt'] = pd.to_datetime(df_viz_review['tanggal'], errors='coerce')
trend = df_viz_review.dropna(subset=['tanggal_dt'])
trend_monthly = trend.set_index('tanggal_dt').resample('ME').size()
if len(trend_monthly) > 0:
    ax3b.plot(trend_monthly.index, trend_monthly.values,
              marker='o', color='#4C9BE8', linewidth=2, markersize=5)
    ax3b.fill_between(trend_monthly.index, trend_monthly.values,
                      alpha=0.15, color='#4C9BE8')
    ax3b.set_xlabel('Bulan')
    ax3b.set_ylabel('Jumlah Review')
    ax3b.set_title('Tren Jumlah Review per Bulan', fontweight='bold')
    ax3b.tick_params(axis='x', rotation=30)
else:
    ax3b.text(0.5, 0.5, 'Data tanggal tidak tersedia', ha='center', va='center',
              transform=ax3b.transAxes)

# ── Plot 4: Rating rata-rata per Sentimen (Bar) ─────────────
ax4b = axes2[1, 0]
avg_rating_sent = df_viz_review.groupby('sentimen')['rating'].mean().sort_values()
colors4b = [PALETTE2.get(s, '#AAAAAA') for s in avg_rating_sent.index]
bars4b = ax4b.bar(avg_rating_sent.index, avg_rating_sent.values,
                  color=colors4b, edgecolor='white')
ax4b.bar_label(bars4b, fmt='%.2f', padding=3, fontsize=10)
ax4b.set_ylim(0, 5.5)
ax4b.set_ylabel('Rata-rata Rating')
ax4b.set_title('Rata-rata Rating per Sentimen', fontweight='bold')

# ── Plot 5: Top 10 Versi Aplikasi (Bar Horizontal) ─────────
ax5b = axes2[1, 1]
versi_counts = (df_viz_review[df_viz_review['versi_aplikasi'] != 'Tidak Diketahui']
                ['versi_aplikasi'].value_counts().head(10))
if len(versi_counts) > 0:
    bars5b = ax5b.barh(versi_counts.index[::-1], versi_counts.values[::-1],
                       color='#9B4CE8', edgecolor='white')
    ax5b.bar_label(bars5b, padding=3, fontsize=9)
    ax5b.set_xlabel('Jumlah Review')
    ax5b.set_title('Top 10 Versi Aplikasi Canva', fontweight='bold')
else:
    ax5b.text(0.5, 0.5, 'Data versi tidak tersedia', ha='center', va='center',
              transform=ax5b.transAxes)

# ── Plot 6: Word Cloud Review Positif ─────────────────────
ax6b = axes2[1, 2]
teks_positif = ' '.join(
    df_viz_review[df_viz_review['sentimen'] == 'Positif']['review_bersih'].dropna()
)
if teks_positif.strip():
    wc = WordCloud(
        width=600, height=350, background_color='white',
        colormap='Greens', max_words=80,
        min_font_size=8
    ).generate(teks_positif)
    ax6b.imshow(wc, interpolation='bilinear')
    ax6b.axis('off')
    ax6b.set_title('Word Cloud — Review Positif', fontweight='bold')
else:
    ax6b.text(0.5, 0.5, 'Tidak ada review positif', ha='center', va='center',
              transform=ax6b.transAxes)

plt.tight_layout()
plt.savefig('visualisasi_review.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Visualisasi review tersimpan: visualisasi_review.png")

## 💾 CELL 11 — Export & Download Semua Output

In [ ]:
# Tutup koneksi DB
conn.close()

# Export CSV bersih
df_survey.to_csv('survei_bersih.csv', index=False, encoding='utf-8-sig')
df_review[['id_review','Username','Rating','Review_Bersih',
           'Likes','Versi_Aplikasi','Tanggal_Str','Sentimen']].to_csv(
    'review_bersih.csv', index=False, encoding='utf-8-sig'
)

print("📦 Mendownload semua file output...")
for fname in ['canva_data.db', 'survei_bersih.csv', 'review_bersih.csv',
              'visualisasi_survei.png', 'visualisasi_review.png']:
    if os.path.exists(fname):
        files.download(fname)
        print(f"  ✅ {fname}")
    else:
        print(f"  ⚠️  {fname} tidak ditemukan")

print("\n🎉 Selesai! Semua output berhasil didownload.")